In [4]:
import os
import numpy as np
import ufl
import basix
import dolfinx
from dolfinx import mesh, fem, io
from dolfinx.io import XDMFFile
from dolfinx.plot import vtk_mesh
from dolfinx.fem.petsc import LinearProblem
from mpi4py import MPI
from petsc4py import PETSc
import pyvista as pv

print(f"dolfinx version: {dolfinx.__version__}")

# Read the mesh
mesh_domain = mesh.create_box(MPI.COMM_WORLD, [np.array([0, 0, 0]), np.array([1, 1, 0.1])],
                         [20, 6, 6], cell_type=mesh.CellType.hexahedron)

tdim = mesh_domain.topology.dim - 1

# Vector function space for displacement field
# element = ufl.VectorElement("CG", mesh_domain.ufl_cell(), 1)
element = basix.ufl.element(family=basix.ufl.ElementFamily.P, cell=basix.ufl.CellType.hexahedron, degree=1, dim=3)
V = fem.FunctionSpace(mesh_domain, element)

# Define trial and test functions
u = ufl.TrialFunction(V)
v = ufl.TestFunction(V)

# Define material properties (Young's modulus, Poisson's ratio)
E = 210e9  # Example: 210 GPa for steel
nu = 0.3
lambda_ = E * nu / ((1 + nu) * (1 - 2 * nu))
mu = E / (2 * (1 + nu))

# Strain and stress
def epsilon(u):
    return ufl.sym(ufl.grad(u))

def sigma(u):
    return lambda_ * ufl.div(u) * ufl.Identity(len(u)) + 2 * mu * epsilon(u)

#### Dirichlet BC (0 displacement on top boundary)
# Define function for displacement boundary conditions
uD = fem.Function(V)
uD.interpolate(lambda x: np.zeros((3, x.shape[1])))     # zero displacements

def boundary_points(x):   # return top boundary points
    return np.isclose(x[1], max(x[1]))

boundary_dofs = fem.locate_dofs_geometrical(V, boundary_points)
bc = fem.dirichletbc(value=uD, dofs=boundary_dofs)

##### Neumann BC (stress/force on the right boundary)
def right_boundary(x):   # return right boundary points
    return np.isclose(x[0], max(x[0]))

load_dofs = mesh.locate_entities_boundary(mesh_domain, tdim, right_boundary)

bound_tag = mesh.MeshTags(load_dofs)

max_load = -6  # Maximum load magnitude
num_increments = 10  # Number of increments
increment = max_load / num_increments  # Load increment

g = fem.Constant(mesh_domain, PETSc.ScalarType((0, 0, 0)))

ds = ufl.Measure('ds', subdomain_data=bound_tag)

# Weak form
a = ufl.inner(sigma(u), epsilon(v)) * ufl.dx
f = fem.Constant(mesh_domain, PETSc.ScalarType((0, 0, 0)))
L = ufl.inner(f, v) * ufl.dx + ufl.inner(g, v) * ufl.ds

# Solve the problem for several steps, and save solutions
problem = dolfinx.fem.petsc.LinearProblem(a, L, bcs=[bc], petsc_options={"ksp_type": "preonly", "pc_type": "lu"})
solutions = []

for step in range(1, num_increments + 1):
    # Update bc applied load value:
    g.value = PETSc.ScalarType((0, step * increment, 0))

    # Solve the problem
    uh = problem.solve()

    # Store the solution for analysis or post-processing
    solutions.append(uh.copy())

# Save results for ParaView
plot_space = fem.FunctionSpace(mesh_domain, ("CG", 1))
u_y = fem.Function(plot_space)

with XDMFFile(mesh_domain.comm, "linEla_debug/linOut.xdmf", "w") as write_file:
    write_file.write_mesh(mesh_domain)
    for step, uh in enumerate(solutions):
        # Plot result to xdmffile
        u_y.name = "$u_y$"
        u_y.interpolate(lambda x: uh.x.array[1::3])  # [1::3] selects y displacements from 3D vector
        write_file.write_function(u_y, step)

        # 2D y displacement pdf plots using pyvista instead
        u_topology, u_cell_types, u_geometry = vtk_mesh(V)
        u_grid = pv.UnstructuredGrid(u_topology, u_cell_types, u_geometry)
        u_grid.point_data["u"] = uh.x.array[1::3]
        u_grid.set_active_scalars("u")
        u_plotter = pv.Plotter()
        u_plotter.add_mesh(u_grid, show_edges=True)
        u_plotter.view_xy()
        u_plotter.save_graphic(f'linEla_debug/linOutCont_y_{step}.pdf')

dolfinx version: 0.9.0


AttributeError: module 'basix.ufl' has no attribute 'ElementFamily'